# Day 2: SQL Analysis
This notebook runs analytical queries on the `bluestock_mf.db` to extract insights.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to Database
db_path = r'../../bluestock_mf.db'
engine = create_engine(f'sqlite:///{db_path}')

def run_query(query):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

## 1. Top 5 Funds by AUM

In [2]:
query1 = """
SELECT scheme_name, aum_crore 
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY aum_crore DESC 
LIMIT 5;
"""
run_query(query1)

,scheme_name,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,49046.0
1,Kotak Emerging Equity Fund - Regular - Growth,47469.0
2,Nippon India Small Cap Fund - Regular - Growth,43630.0
3,DSP Top 100 Equity Fund - Regular - Growth,41828.0
4,UTI Mid Cap Fund - Regular - Growth,41728.0


## 2. SIP Year-over-Year (YoY) Growth

In [3]:
query2 = """
WITH YearlySIP AS (
    SELECT d.year, SUM(amount_inr) as total_sip
    FROM fact_transactions t
    JOIN dim_date d ON t.transaction_date = d.date
    WHERE t.transaction_type = 'SIP'
    GROUP BY d.year
)
SELECT curr.year, curr.total_sip, 
       ((curr.total_sip - prev.total_sip) * 100.0 / prev.total_sip) as yoy_growth_pct
FROM YearlySIP curr
LEFT JOIN YearlySIP prev ON curr.year = prev.year + 1;
"""
run_query(query2)

,year,total_sip,yoy_growth_pct
0,2024,61026046.0,NaN
1,2025,26279263.0,-56.937628


## 3. Transaction Amount by State

In [4]:
query3 = """
SELECT state, SUM(amount_inr) as total_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_amount DESC
LIMIT 10;
"""
run_query(query3)

,state,total_amount
0,Tamil Nadu,126020641.0
1,Madhya Pradesh,124230586.0
2,West Bengal,121974095.0
3,Punjab,121295845.0
4,Uttar Pradesh,119497728.0
5,Gujarat,117945641.0
6,Delhi,116939568.0
7,Rajasthan,114352455.0
8,Karnataka,113816812.0
9,Haryana,112280777.0


## 4. High Rated Funds with 3-Year Returns

In [5]:
query4 = """
SELECT scheme_name, return_3yr_pct, morningstar_rating
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
WHERE morningstar_rating >= 4
ORDER BY return_3yr_pct DESC
LIMIT 10;
"""
run_query(query4)

,scheme_name,return_3yr_pct,morningstar_rating
0,SBI Small Cap Fund - Regular Plan - Growth,23.39,5
1,SBI Small Cap Fund - Direct Plan - Growth,23.14,4
2,ABSL Small Cap Fund - Regular - Growth,22.38,5
3,Axis Small Cap Fund - Regular - Growth,20.98,4
4,Nippon India Small Cap Fund - Regular - Growth,20.15,4
5,DSP Small Cap Fund - Regular - Growth,20.08,4
6,Kotak Emerging Equity Fund - Regular - Growth,18.23,4
7,DSP Midcap Fund - Regular - Growth,17.16,4
8,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,16.58,5
9,Kotak Flexicap Fund - Regular - Growth,15.65,5
